This step is simply to go back where ropt is defined. Since python doesn't allow us to import from backwards folders.

In [1]:
from os import getcwd, chdir

if getcwd().split('/')[-1] == 'composite_indicator':
    chdir('..')
getcwd()

'/mnt/c/Users/kosta/Desktop/wsl/ROpt'

# Reading the data

The example data given contains a few columns that are not numeric. We'll remove them since we don't have a clear way to translate them here.

In [2]:
from pandas import read_csv, read_excel, Series

In [3]:
raw_data = read_excel('composite_indicator/ODS7_Data.xlsx')
raw_data = raw_data[raw_data.columns[1:]]

# for col in raw_data.columns:
#     raw_data[col] = raw_data[col].astype(str)
#     raw_data[col] = raw_data[col].apply(lambda x: x.replace(',','.'))
#     raw_data[col] = raw_data[col].astype(float)

raw_data

,ODS_7_1,ODS_7_2,ODS_7_3,ODS_7_4,ODS_7_5
0,1.000000,0.131628,0.118105,0.200963,0.397214
1,1.000000,0.007386,0.159123,0.000182,0.003373
2,0.972387,0.400686,0.195898,0.000016,0.286295
3,0.998028,0.169480,0.159830,0.002743,0.119727
4,1.000000,0.632683,0.156294,1.000000,1.000000
5,1.000000,0.319968,0.128713,0.288470,0.110530
6,1.000000,0.397520,0.067893,0.189866,0.307464
7,1.000000,0.443946,0.044554,0.024619,0.072316
8,1.000000,0.235954,0.000000,0.001644,0.083800
9,1.000000,0.111448,0.083451,0.000008,0.000000


In [4]:
from ropt.normalizer import robust_normalize, outlier_removal, series_ranking

reference = raw_data[raw_data.columns[-1]]
data = robust_normalize(outlier_removal(raw_data),reference_series=reference)
data=data[data.columns[:-1]]
# data = data[data.columns[:3].tolist() + data.columns[4:].tolist()]
# reference = data[data.columns[3]]
data, reference

(     ODS_7_1   ODS_7_2   ODS_7_3   ODS_7_4
 0   1.000000  0.131628  0.327130  1.000000
 1   1.000000  0.007386  0.440744  0.000913
 2   0.852632  0.400686  0.542605  0.000079
 3   0.989474  0.169480  0.442703  0.013776
 4   1.000000  0.632683  0.432909  1.000000
 5   1.000000  0.319968  0.356513  1.000000
 6   1.000000  0.397520  0.188051  0.953748
 7   1.000000  0.443946  0.123408  0.123670
 8   1.000000  0.235954  0.000000  0.008258
 9   1.000000  0.111448  0.231146  0.000040
 10  0.800000  0.166579  0.135162  0.349849
 11  1.000000  0.212477  0.321254  0.365571
 12  1.000000  0.239910  0.333007  0.400786
 13  0.389474  0.124374  0.303624  0.000000
 14  0.905263  0.822342  0.425073  0.018580
 15  0.263158  0.100501  0.252693  0.330792
 16  0.000000  1.000000  0.844270  0.011474
 17  0.410526  0.577816  0.538688  0.995514
 18  1.000000  0.108547  0.538688  0.001390
 19  1.000000  0.169612  0.331048  0.005677
 20  0.000000  0.653785  0.479922  0.104177
 21  0.473684  0.337510  0.05876

In [5]:
data

,ODS_7_1,ODS_7_2,ODS_7_3,ODS_7_4
0,1.000000,0.131628,0.327130,1.000000
1,1.000000,0.007386,0.440744,0.000913
2,0.852632,0.400686,0.542605,0.000079
3,0.989474,0.169480,0.442703,0.013776
4,1.000000,0.632683,0.432909,1.000000
5,1.000000,0.319968,0.356513,1.000000
6,1.000000,0.397520,0.188051,0.953748
7,1.000000,0.443946,0.123408,0.123670
8,1.000000,0.235954,0.000000,0.008258
9,1.000000,0.111448,0.231146,0.000040


In [6]:
reference

0     0.397214
1     0.003373
2     0.286295
3     0.119727
4     1.000000
5     0.110530
6     0.307464
7     0.072316
8     0.083800
9     0.000000
10    0.196036
11    0.359326
12    0.322346
13    0.036778
14    0.016778
15    0.006728
16    0.007131
17    0.219787
18    0.112544
19    0.292508
20    0.011957
21    0.074030
22    0.155008
23    0.232865
24    0.368007
25    0.001993
26    0.130792
27    0.717789
Name: ODS_7_5, dtype: float64

In [7]:
from ropt.multi_objective import multi_objective_optimization, fix_coef
from util import BOD_Calculation, Entropy_Calculation, PCA_Calculation, Minimal_Uncertainty
from ropt.means import normal_weighted_arithimetic_mean
from objectives import MaxCorrel, PCA, MaxEntropy, MinContinuousUncertanty, MinUncertanty
from methods import equal_weights, specialist_weights
from ropt.optimizer import optimization
from ropt.temperature_optimizer import temperature_clock_optimization
from ropt.ray_optimizer import ray_clock_optimization
from ropt.swarm_optimizer import swarm_optimization

In [8]:
o_maxes= []; a_maxes= []
o_mines= []; a_mines= []
alternative_methods = [
    Series([r.ci for r in BOD_Calculation(data).run()]),
    Series([r.ci for r in Entropy_Calculation(data).run()]),
    Series([r.ci for r in PCA_Calculation(data).run()]),
    equal_weights(data)
]

In [ ]:
Series([r.ci for r in Minimal_Uncertainty(data, alternative_methods).run()])

In [9]:
o_max= []; a_max= []
o_min= []; a_min= []
true_objectives = [
    MaxCorrel(reference),
    PCA(data),
    MaxEntropy(),
    MinUncertanty(alternative_methods)
]
swarm_answer = multi_objective_optimization(
    data, true_objectives, 
    thresholds=[1 for _ in true_objectives], limit_mquality=0.00000001,
    out_max=o_max, out_min=o_min, ans_max=a_max, ans_min=a_min,
    optimizer=swarm_optimization(
        limit_nstep=100000, limit_dquality = 0.00000001
    )    
)
o_maxes.append(o_max); o_mines.append(o_min)
a_maxes.append(a_max); a_mines.append(a_min)

print(['%.3f'%val for val in swarm_answer])

Found best quality
0.6948717948717948
['0.248', '0.326', '0.000', '0.426']


In [10]:
print(['%.3f'%val for val in swarm_answer])

['0.248', '0.326', '0.000', '0.426']


In [11]:
o_max= []; a_max= []
o_min= []; a_min= []
true_objectives = [
    MaxCorrel(reference),
    PCA(data),
    MaxEntropy(),
    MinUncertanty(alternative_methods)
]
casted_answer = multi_objective_optimization(
    data, true_objectives, 
    thresholds=[1 for _ in true_objectives], limit_mquality=0.00000001,
    out_max=o_max, out_min=o_min, ans_max=a_max, ans_min=a_min,
    optimizer=ray_clock_optimization(
        dstep=0.003, limit_nstep=100000,
        limit_dquality = 0.00000001
    )    
)
o_maxes.append(o_max); o_mines.append(o_min)
a_maxes.append(a_max); a_mines.append(a_min)

print(['%.3f'%val for val in casted_answer])

Found best quality
0.6820512820512821
['0.455', '0.076', '0.094', '0.375']


In [12]:
o_max= []; a_max= []
o_min= []; a_min= []
true_objectives = [
    MaxCorrel(reference),
    PCA(data),
    MaxEntropy(),
    MinUncertanty(alternative_methods)
]
anealled_answer = multi_objective_optimization(
    data, true_objectives, 
    thresholds=[1 for _ in true_objectives], limit_mquality=0.00000001,
    out_max=o_max, out_min=o_min, ans_max=a_max, ans_min=a_min,
    optimizer=temperature_clock_optimization(
        dstep=0.003, limit_nstep=100000,
        limit_dquality = 0.00000001
    )    
)
o_maxes.append(o_max); o_mines.append(o_min)
a_maxes.append(a_max); a_mines.append(a_min)

print(['%.3f'%val for val in anealled_answer])

Found best quality
0.6872666474092429
['0.045', '0.347', '0.304', '0.304']


In [13]:
o_max= []; a_max= []
o_min= []; a_min= []
true_objectives = [
    MaxCorrel(reference),
    PCA(data),
    MaxEntropy(),
    MinUncertanty(alternative_methods)
]
true_answer = multi_objective_optimization(
    data, true_objectives, 
    thresholds=[1 for _ in true_objectives], limit_mquality=0.000000001,
    out_max=o_max, out_min=o_min, ans_max=a_max, ans_min=a_min,
    optimizer=optimization(
        dstep=0.003, limit_nstep=100000,
        limit_dquality = 0.00000001
    )    
)
o_maxes.append(o_max); o_mines.append(o_min)
a_maxes.append(a_max); a_mines.append(a_min)

print(['%.3f'%val for val in true_answer])

Found best quality
0.5818881226412634
['0.358', '0.055', '0.187', '0.400']


In [14]:
o_max= []; a_max= []
o_min= []; a_min= []
objectives = [
    MaxCorrel(reference),
    PCA(data),
    MaxEntropy(),
    MinUncertanty(alternative_methods),
    MinContinuousUncertanty(alternative_methods)
]
simulated_answer = multi_objective_optimization(
    data, objectives, 
    importance_factors=[2,2,2,2,1],
    thresholds=[1 for _ in objectives], limit_mquality=0.000000001,
    out_max=o_max, out_min=o_min, ans_max=a_max, ans_min=a_min,
    optimizer=optimization(
        dstep=0.003, limit_nstep=100000,
        limit_dquality = 0.00000001
    )
)
o_maxes.append(o_max); o_mines.append(o_min)
a_maxes.append(a_max); a_mines.append(a_min)

print(['%.3f'%val for val in simulated_answer])

Found best quality
0.33859378727097406
['0.358', '0.055', '0.187', '0.400']


In [15]:
o_max= []; a_max= []
o_min= []; a_min= []
changed_objectives = [
    MaxCorrel(reference),
    PCA(data),
    MaxEntropy(),
    MinContinuousUncertanty(alternative_methods)
]
changed_answer = multi_objective_optimization(
    data, changed_objectives, 
    thresholds=[1 for _ in changed_objectives], limit_mquality=0.000000001,
    out_max=o_max, out_min=o_min, ans_max=a_max, ans_min=a_min,
    optimizer=optimization(
        dstep=0.003, limit_nstep=100000,
        limit_dquality = 0.00000001
    )
)
o_maxes.append(o_max); o_mines.append(o_min)
a_maxes.append(a_max); a_mines.append(a_min)

print(['%.3f'%val for val in simulated_answer])

No more different points
0.7432345117626059
['0.358', '0.055', '0.187', '0.400']


In [16]:
indexes = [
    [0,1,2,3],
    [0,1,2,3],
    [0,1,2,3],
    [0,1,2,3,4],
    [0,1,2,4]
]
f_max=[float('-inf') for _ in range(5)];f_min=[float('inf') for _ in range(5)];fa_max=[[] for _ in range(5)];fa_min=[[] for _ in range(5)]

In [17]:
for ids,omas,omis,amas,amis in zip(indexes,o_maxes,o_mines,a_maxes,a_mines):
    for i,oma,omi,ama,ami in zip(ids,omas,omis,amas,amis):
        if f_max[i] < oma:
            f_max[i] = oma
            fa_max[i] = ama
        if f_min[i] > omi:
            f_min[i] = omi
            fa_min[i] = ami

In [18]:
simulated_answer

[0.3580000000000001,
 0.05499999999999983,
 0.18699999999999994,
 0.40000000000000013]

In [19]:
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(simulated_answer, col), axis=1)
# for v in ci:
#     print(v)
# ci

In [20]:
qualities = [objective(ci) for objective in objectives]
for mi, val, ma in zip(f_min,qualities,f_max):
    print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])

['          0.042', '          0.588', '          0.614', '          0.955']
['          0.170', '          0.244', '          0.297', '          0.582']
['          0.382', '          0.861', '          0.940', '          0.859']
['         -0.606', '         -0.267', '         -0.108', '          0.682']
['         -0.490', '         -0.328', '         -0.108', '          0.425']


In [21]:
anealled_answer

[0.04506249999999981,
 0.3471250000000001,
 0.30400000000000005,
 0.30381250000000004]

In [22]:
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(anealled_answer, col), axis=1)
# for v in ci:
#     print(v)
# ci

In [23]:
qualities = [objective(ci) for objective in objectives]
for mi, val, ma in zip(f_min,qualities,f_max):
    print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])

['          0.042', '          0.435', '          0.614', '          0.687']
['          0.170', '          0.258', '          0.297', '          0.688']
['          0.382', '          0.819', '          0.940', '          0.783']
['         -0.606', '         -0.224', '         -0.108', '          0.767']
['         -0.490', '         -0.390', '         -0.108', '          0.261']


In [24]:
swarm_answer

[0.24822329976533794, 0.3259658024405293, 0, 0.42581089779413284]

In [25]:
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(swarm_answer, col), axis=1)
# for v in ci:
#     print(v)
# ci

In [26]:
qualities = [objective(ci) for objective in objectives]
for mi, val, ma in zip(f_min,qualities,f_max):
    print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])

['          0.042', '          0.493', '          0.614', '          0.788']
['          0.170', '          0.259', '          0.297', '          0.700']
['          0.382', '          0.777', '          0.940', '          0.708']
['         -0.606', '         -0.260', '         -0.108', '          0.695']
['         -0.490', '         -0.333', '         -0.108', '          0.410']


In [27]:
casted_answer

[0.4545519151687625,
 0.07642294692993142,
 0.09420013046264636,
 0.37482500743865976]

In [28]:
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(casted_answer, col), axis=1)
# for v in ci:
#     print(v)
# ci

In [29]:
qualities = [objective(ci) for objective in objectives]
for mi, val, ma in zip(f_min,qualities,f_max):
    print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])

['          0.042', '          0.564', '          0.614', '          0.912']
['          0.170', '          0.257', '          0.297', '          0.684']
['          0.382', '          0.849', '          0.940', '          0.838']
['         -0.606', '         -0.267', '         -0.108', '          0.682']
['         -0.490', '         -0.365', '         -0.108', '          0.327']


In [30]:
true_answer

[0.3580000000000001,
 0.05499999999999983,
 0.18699999999999994,
 0.40000000000000013]

In [31]:
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(true_answer, col), axis=1)
# for v in ci:
#     print(v)
# ci

In [32]:
qualities = [objective(ci) for objective in objectives]
for mi, val, ma in zip(f_min,qualities,f_max):
    print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])

['          0.042', '          0.588', '          0.614', '          0.955']
['          0.170', '          0.244', '          0.297', '          0.582']
['          0.382', '          0.861', '          0.940', '          0.859']
['         -0.606', '         -0.267', '         -0.108', '          0.682']
['         -0.490', '         -0.328', '         -0.108', '          0.425']


In [33]:
changed_answer

[0.3827773437499996,
 0.11821875000000026,
 0.011484375000000165,
 0.48751953124999947]

In [34]:
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(changed_answer, col), axis=1)
# for v in ci:
#     print(v)
# ci

In [35]:
qualities = [objective(ci) for objective in objectives]
for mi, val, ma in zip(f_min,qualities,f_max):
    print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])

['          0.042', '          0.536', '          0.614', '          0.862']
['          0.170', '          0.265', '          0.297', '          0.743']
['          0.382', '          0.838', '          0.940', '          0.817']
['         -0.606', '         -0.325', '         -0.108', '          0.564']
['         -0.490', '         -0.365', '         -0.108', '          0.328']


# UNCERTANTY START

In [36]:
uncertanty_answers = []

In [37]:
ans = swarm_optimization(
    limit_nstep=100000, limit_dquality = 0.00001,
)(
    data, lambda ci: MinUncertanty(alternative_methods)(ci),
)
uncertanty_answers.append(ans)
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
quality = MinUncertanty(alternative_methods)(ci)
ans, sum(ans), quality

([0.2637335075109574,
  0.24018056002263546,
  0.27275867634277073,
  0.22332725612363644],
 1.0,
 -0.11734693877551021)

In [38]:
ans = ray_clock_optimization(
    dstep=0.003, limit_nstep=100000,
    limit_dquality = 0.00000001
)(
    data, lambda ci: MinUncertanty(alternative_methods)(ci),
)
uncertanty_answers.append(ans)
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
quality = MinUncertanty(alternative_methods)(ci)
ans, sum(ans), quality

([0.25, 0.25, 0.25, 0.25], 1.0, -0.10841836734693877)

In [39]:
ans = temperature_clock_optimization(
    dstep=0.003, limit_nstep=100000,
    limit_dquality = 0.00000001
)(
    data, lambda ci: MinUncertanty(alternative_methods)(ci),
)
uncertanty_answers.append(ans)
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
quality = MinUncertanty(alternative_methods)(ci)
ans, sum(ans), quality

([0.253, 0.262, 0.241, 0.244], 1.0, -0.11096938775510204)

In [40]:
ans = optimization(
    dstep=0.003, limit_nstep=100000,
    limit_dquality = 0.00000001
)(
    data, lambda ci: MinUncertanty(alternative_methods)(ci),
)
uncertanty_answers.append(ans)
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
quality = MinUncertanty(alternative_methods)(ci)
ans, sum(ans), quality

([0.25, 0.25, 0.25, 0.25], 1.0, -0.10841836734693877)

In [41]:
for ans in uncertanty_answers:
    ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
    qualities = [objective(ci) for objective in objectives]
    for mi, val, ma in zip(f_min,qualities,f_max):
        print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])
    print('-'*20)

['          0.042', '          0.582', '          0.614', '          0.943']
['          0.170', '          0.208', '          0.297', '          0.297']
['          0.382', '          0.873', '          0.940', '          0.881']
['         -0.606', '         -0.117', '         -0.108', '          0.982']
['         -0.490', '         -0.239', '         -0.108', '          0.658']
--------------------
['          0.042', '          0.578', '          0.614', '          0.936']
['          0.170', '          0.215', '          0.297', '          0.350']
['          0.382', '          0.865', '          0.940', '          0.867']
['         -0.606', '         -0.108', '         -0.108', '          1.000']
['         -0.490', '         -0.224', '         -0.108', '          0.697']
--------------------
['          0.042', '          0.574', '          0.614', '          0.929']
['          0.170', '          0.215', '          0.297', '          0.349']
['          0.382', '          0.8

In [42]:
max_uncertanty_answers = []

In [43]:
ans = swarm_optimization(
    limit_nstep=100000, limit_dquality = 0.00001,
)(
    data, lambda ci: -MinUncertanty(alternative_methods)(ci),
)
max_uncertanty_answers.append(ans)
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
quality = MinUncertanty(alternative_methods)(ci)
ans, sum(ans), quality

([0, 0, 0.9973451903138514, 0.002654809686148507],
 0.9999999999999999,
 -0.6020408163265306)

In [44]:
ans = ray_clock_optimization(
    dstep=0.003, limit_nstep=100000,
    limit_dquality = 0.00000001
)(
    data, lambda ci: -MinUncertanty(alternative_methods)(ci),
)
max_uncertanty_answers.append(ans)
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
quality = MinUncertanty(alternative_methods)(ci)
ans, sum(ans), quality

([0.006438544034957886, 0.0, 0.9907413787841798, 0.0028200771808624273],
 1.0000000000000002,
 -0.6058673469387755)

In [45]:
ans = temperature_clock_optimization(
    dstep=0.003, limit_nstep=100000,
    limit_dquality = 0.00000001
)(
    data, lambda ci: -MinUncertanty(alternative_methods)(ci),
)
max_uncertanty_answers.append(ans)
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
quality = MinUncertanty(alternative_methods)(ci)
ans, sum(ans), quality

([0.051999999999999824,
  0.31900000000000006,
  0.15999999999999992,
  0.4690000000000002],
 1.0,
 -0.27933673469387754)

In [46]:
ans = optimization(
    dstep=0.003, limit_nstep=100000,
    limit_dquality = 0.00000001
)(
    data, lambda ci: -MinUncertanty(alternative_methods)(ci),
)
max_uncertanty_answers.append(ans)
ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
quality = MinUncertanty(alternative_methods)(ci)
ans, sum(ans), quality

([0.253, 0.247, 0.25, 0.25], 1.0, -0.11096938775510204)

In [47]:
for ans in max_uncertanty_answers:
    ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
    qualities = [objective(ci) for objective in objectives]
    for mi, val, ma in zip(f_min,qualities,f_max):
        print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])
    print('-'*20)

['          0.042', '          0.186', '          0.614', '          0.252']
['          0.170', '          0.264', '          0.297', '          0.737']
['          0.382', '          0.758', '          0.940', '          0.675']
['         -0.606', '         -0.602', '         -0.108', '          0.008']
['         -0.490', '         -0.566', '         -0.108', '         -0.200']
--------------------
['          0.042', '          0.189', '          0.614', '          0.258']
['          0.170', '          0.263', '          0.297', '          0.733']
['          0.382', '          0.757', '          0.940', '          0.672']
['         -0.606', '         -0.606', '         -0.108', '          0.000']
['         -0.490', '         -0.562', '         -0.108', '         -0.188']
--------------------
['          0.042', '          0.450', '          0.614', '          0.713']
['          0.170', '          0.263', '          0.297', '          0.728']
['          0.382', '          0.7

In [48]:
for ans in fa_max:
    ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
    qualities = [objective(ci) for objective in objectives]
    for mi, val, ma in zip(f_min,qualities,f_max):
        print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])
    print('-'*20)

['          0.042', '          0.614', '          0.614', '          1.000']
['          0.170', '          0.216', '          0.297', '          0.360']
['          0.382', '          0.891', '          0.940', '          0.912']
['         -0.606', '         -0.228', '         -0.108', '          0.759']
['         -0.490', '         -0.288', '         -0.108', '          0.530']
--------------------
['          0.042', '          0.131', '          0.614', '          0.155']
['          0.170', '          0.297', '          0.297', '          1.000']
['          0.382', '          0.574', '          0.940', '          0.345']
['         -0.606', '         -0.421', '         -0.108', '          0.372']
['         -0.490', '         -0.477', '         -0.108', '          0.035']
--------------------
['          0.042', '          0.437', '          0.614', '          0.690']
['          0.170', '          0.233', '          0.297', '          0.496']
['          0.382', '          0.9

In [49]:
for ans in fa_min:
    ci = data.apply(lambda col: normal_weighted_arithimetic_mean(ans, col), axis=1)
    qualities = [objective(ci) for objective in objectives]
    for mi, val, ma in zip(f_min,qualities,f_max):
        print(['%15.3f'%v for v in (mi, val, ma, (val-mi)/(ma-mi))])
    print('-'*20)

['          0.042', '          0.042', '          0.614', '          0.000']
['          0.170', '          0.283', '          0.297', '          0.886']
['          0.382', '          0.689', '          0.940', '          0.550']
['         -0.606', '         -0.490', '         -0.108', '          0.233']
['         -0.490', '         -0.613', '         -0.108', '         -0.322']
--------------------
['          0.042', '          0.349', '          0.614', '          0.537']
['          0.170', '          0.170', '          0.297', '          0.000']
['          0.382', '          0.806', '          0.940', '          0.759']
['         -0.606', '         -0.366', '         -0.108', '          0.482']
['         -0.490', '         -0.365', '         -0.108', '          0.327']
--------------------
['          0.042', '          0.420', '          0.614', '          0.661']
['          0.170', '          0.263', '          0.297', '          0.731']
['          0.382', '          0.5